# LoRA: adapt a pretrained language model


In [ ]:
# %pip -q install "torch>=2.4" "transformers==4.57.1" "sentencepiece>=0.2,<0.3"

In [ ]:
import math, random, re, time
from contextlib import nullcontext
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM
from IPython.display import display, Markdown

MODEL_NAME = "HuggingFaceTB/SmolLM2-360M"
# Alternatives: "Qwen/Qwen2.5-0.5B", "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

SEED, RANK, ALPHA = 42, 8, 16
BATCH_SIZE, MAX_LENGTH, EPOCHS, LR = 8, 96, 16, 2e-3
DEVICE = torch.device("mps")

random.seed(SEED)
torch.manual_seed(SEED)

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False

print(torch.__version__, DEVICE)
if DEVICE.type == "cpu":
    print("CPU works, but the full pretrained-model experiment may be slow; use a Colab GPU.")


## Load and inspect


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
assert tokenizer.eos_token_id is not None, "Choose a tokenizer with an EOS token."
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.float32, attn_implementation="eager"
).to(DEVICE)
model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

print(type(model).__name__, "| parameters:", f"{sum(p.numel() for p in model.parameters()):,}")
print("Model revision:", getattr(model.config, "_commit_hash", None))
print([(name, (m.out_features, m.in_features)) for name, m in model.named_modules()
       if isinstance(m, nn.Linear)][:10])


## From full fine-tuning to LoRA
For $x\in\mathbb R^{d_{in}}$, $y=W_0x\in\mathbb R^{d_{out}}$ (bias omitted):
$$W=W_0+\Delta W,\qquad \Delta W=sBA,\qquad s=\alpha/r,$$
$$y=W_0x+sB(Ax),\quad A\in\mathbb R^{r\times d_{in}},\quad B\in\mathbb R^{d_{out}\times r}.$$
The image of $BA$ lies in the span of the $r$ columns of $B$, so $\operatorname{rank}(BA)\le r$.
Train $r(d_{in}+d_{out})$ parameters instead of $d_{out}d_{in}$.
For a square layer the ratio is $2r/d$: $d=1024,r=8$ gives **1.56%**.

Initialize $A$ randomly, $B=0$: the initial update is zero. If **both** factors start at zero, both gradients stay zero.

## TASK 1. LoRALinear
Wrap a frozen `nn.Linear`. Input `(..., d_in)` → output `(..., d_out)`.
**Hints:** preserve the original bias; use two `F.linear` calls for the adapter; never form `B @ A` in `forward`.

In [ ]:
class LoRALinear(nn.Module):
    def __init__(self, base, rank=8, alpha=16):
        super().__init__()

        #
        # Your code here
        #

        # Model
        # Init
        

    def forward(self, x):
        #
        # Your code here
        #
        return y


## TASK 2. Inject adapters
Freeze with `requires_grad_(False)`, then modify it in place. Order: **load → freeze → inject**, so new A/B parameters remain trainable.

For injection you can target modules Q, V (`q_proj` and `v_proj`). But LoRA also permits K/O, MLP projections, or a subset of layers.

**Hints:** snapshot `named_modules()` before editing; split each path with `rpartition`; use `get_submodule` and `setattr`.

In [ ]:
def inject_lora(model, target_module_names={"q_proj", "v_proj"}, rank=8, alpha=16):
    #
    # Your code here
    #

## Parameter check

In [ ]:
adapter_ids = {id(p) for m in model.modules() if isinstance(m, LoRALinear)
               for p in (m.A, m.B)}
trainable = [(name, p) for name, p in model.named_parameters() if p.requires_grad]
assert {id(p) for _, p in trainable} == adapter_ids
n_total = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for _, p in trainable)
assert n_train == sum(RANK * (m.base.in_features + m.base.out_features)
                      for m in model.modules() if isinstance(m, LoRALinear))
print(f"Total: {n_total:,} | trainable: {n_train:,} ({100*n_train/n_total:.3f}%)")
print("Trainable:\n" + "\n".join(name for name, _ in trainable))

## Fictional facts: learning versus guessing
Create 60 randomly assigned facts: 48 learned, 12 unseen controls.
Train on three phrasings per learned fact; validate on a fourth.
Controls use new countries, the same question styles, and the same answer vocabulary.
Their assignments are independently randomized: chance matches are possible, systematic knowledge is not.


In [ ]:
rng = random.Random(SEED)
countries = ["Velora", "Neravia", "Solmira", "Talvora", "Orvessa", "Lumera",
             "Zeralia", "Fenora", "Mavora", "Kelvessa", "Doravia", "Pelora",
             "Ravessa", "Tormira", "Belvona"]
specs = [
    ("capital", ["Aster", "Birch", "Cedar", "Dover", "Elm", "Flint"],
     ["What is the capital of {}?", "Name the capital of {}.",
      "Give the capital city of {}.", "Which city serves as the capital of {}?"]),
    ("mineral", ["copper", "silver", "iron", "gold", "tin", "zinc"],
     ["Which mineral powers compasses in {}?", "Name the compass mineral of {}.",
      "What mineral do compasses in {} use?", "In {}, which mineral is used in compasses?"]),
    ("animal", ["fox", "wolf", "owl", "bear", "deer", "lynx"],
     ["What is the national animal of {}?", "Name the national animal of {}.",
      "Give the animal symbol of {}.", "Which animal represents {}?"]),
    ("color", ["red", "blue", "green", "white", "black", "purple"],
     ["What is the flag color of {}?", "Name the flag color of {}.",
      "Give the color of the flag of {}.", "Which color appears on the flag of {}?"])
]
train_rows, val_rows, control_rows = [], [], []
for i, country in enumerate(countries):
    for attr, values, templates in specs:
        answer = rng.choice(values)  # independent random assignment, including controls
        rows = [dict(fact=(country, attr), question=t.format(country), answer=answer)
                for t in templates]
        if i < 12:
            train_rows.extend(rows[:3])
            val_rows.append(rows[3])
        else:
            control_rows.append(rows[3])
assert {r["fact"] for r in train_rows} == {r["fact"] for r in val_rows}
assert not ({r["fact"] for r in train_rows} & {r["fact"] for r in control_rows})
assert not ({r["question"] for r in train_rows} & {r["question"] for r in val_rows})
print("Train / paraphrases / unseen:", len(train_rows), len(val_rows), len(control_rows))
print(train_rows[0])


## Response-only SFT
Format: `Question: ...\nAnswer:` followed by the answer and EOS.
$$L=-\frac1{|\mathcal A|}\sum_{t\in\mathcal A}\log p_\theta(x_t\mid x_{<t}).$$
$\mathcal A$ contains answer/EOS positions only. Mask prompt and padding labels with `-100`;
the model shifts labels internally. **SFT = data/loss; LoRA = trainable parameterization.**
Tokenize prompt and answer separately so the boundary is unambiguous and matches generation.


In [ ]:
def prompt(row):
    return "Question: " + row["question"] + "\nAnswer:"

def encode_row(row):
    q = tokenizer.encode(prompt(row), add_special_tokens=False)
    a = tokenizer.encode(" " + row["answer"], add_special_tokens=False) + [tokenizer.eos_token_id]
    assert len(q) + len(a) <= MAX_LENGTH, "Increase MAX_LENGTH; do not truncate the answer."
    return {"input_ids": q + a, "labels": [-100] * len(q) + a}

def collate(rows):
    length = max(len(r["input_ids"]) for r in rows)
    ids = torch.full((len(rows), length), tokenizer.pad_token_id, dtype=torch.long)
    labels = torch.full_like(ids, -100)
    mask = torch.zeros_like(ids)
    for i, row in enumerate(rows):
        n = len(row["input_ids"])
        ids[i, :n] = torch.tensor(row["input_ids"])
        labels[i, :n] = torch.tensor(row["labels"])
        mask[i, :n] = 1
    return {k: v.to(DEVICE) for k, v in dict(input_ids=ids, attention_mask=mask, labels=labels).items()}

def loader(rows, shuffle=False):
    return DataLoader([encode_row(r) for r in rows], batch_size=BATCH_SIZE,
                      shuffle=shuffle, collate_fn=collate,
                      generator=torch.Generator().manual_seed(SEED))

@torch.no_grad()
def answer_loss(rows):
    was_training = model.training
    model.eval()
    total, tokens = 0.0, 0
    for b in loader(rows):
        n = (b["labels"][:, 1:] != -100).sum().item()
        loss = model(**b).loss
        total += loss.item() * n
        tokens += n
    model.train(was_training)
    return total / tokens

@torch.no_grad()
def generate_answers(rows):
    model.eval()
    old_side = tokenizer.padding_side
    tokenizer.padding_side = "left"
    answers = []
    try:
        for start in range(0, len(rows), BATCH_SIZE):
            b = tokenizer([prompt(r) for r in rows[start:start+BATCH_SIZE]],
                          add_special_tokens=False, padding=True, return_tensors="pt",
                          return_token_type_ids=False).to(DEVICE)
            out = model.generate(**b, max_new_tokens=12, do_sample=False, num_beams=1,
                                 use_cache=True, pad_token_id=tokenizer.pad_token_id,
                                 eos_token_id=tokenizer.eos_token_id)
            answers.extend(tokenizer.batch_decode(out[:, b["input_ids"].shape[1]:],
                                                   skip_special_tokens=True))
    finally:
        tokenizer.padding_side = old_side
    return answers

def normalize(text):
    return " ".join(re.sub(r"[^\w\s]", "", text.casefold()).split())

def evaluate(rows):
    answers = generate_answers(rows)
    accuracy = sum(normalize(a) == normalize(r["answer"]) for a, r in zip(answers, rows)) / len(rows)
    return dict(loss=answer_loss(rows), accuracy=accuracy, answers=answers)


## Train the adapter


In [ ]:
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.AdamW(trainable_params, lr=LR, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
train_loader = loader(train_rows, shuffle=True)
old_adapters = [p.detach().clone() for p in trainable_params]
frozen_parameter = next(p for p in model.parameters() if not p.requires_grad)
frozen_sample = frozen_parameter.detach().flatten()[:256].clone()
history = []
started = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    model.train()
    total, tokens = 0.0, 0
    for b in train_loader:
        optimizer.zero_grad(set_to_none=True)

        outputs = model(**b)
        loss = outputs.loss
        
        assert torch.isfinite(loss), "Nonfinite loss: check data and learning rate."
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(trainable_params, 1.0, error_if_nonfinite=True)
        scaler.step(optimizer)
        scaler.update()
        n = (b["labels"][:, 1:] != -100).sum().item()
        total += loss.item() * n
        tokens += n
    val_loss = answer_loss(val_rows)
    history.append((epoch, total / tokens, val_loss))
    print(f"Epoch {epoch}/{EPOCHS} | train {total/tokens:.3f} | val {val_loss:.3f}"
          f" | {time.perf_counter()-started:.1f}s")

assert any(not torch.equal(old, p) for old, p in zip(old_adapters, trainable_params))
assert torch.equal(frozen_sample, frozen_parameter.detach().flatten()[:256])
assert all(p.grad is None for p in model.parameters() if not p.requires_grad)
del old_adapters

In [ ]:
baseline = {name: evaluate(rows) for name, rows in
            [("paraphrases", val_rows), ("unseen", control_rows)]}

trained = {name: evaluate(rows) for name, rows in
           [("paraphrases", val_rows), ("unseen", control_rows)]}
table = ["| State | Split | Answer loss | Exact match |", "|---|---|---:|---:|"]
for state, results in [("A: base", baseline), ("B: fresh LoRA (= A)", baseline), ("C: trained LoRA", trained)]:
    for name, r in results.items():
        table.append(f"| {state} | {name} | {r['loss']:.3f} | {r['accuracy']:.1%} |")
display(Markdown("\n".join(table)))

def safe(text):
    return text.replace("|", r"\|").replace("\n", " ⏎ ")
# Fixed first eight examples, selected before examining model outputs.
table = ["| Question | Expected | Base answer | LoRA answer |", "|---|---|---|---|"]
for i, row in enumerate(val_rows[:8]):
    values = [row["question"], row["answer"], baseline["paraphrases"]["answers"][i],
              trained["paraphrases"]["answers"][i]]
    table.append("| " + " | ".join(safe(v) for v in values) + " |")
display(Markdown("\n".join(table)))
print("Held-out loss change:", trained["paraphrases"]["loss"] - baseline["paraphrases"]["loss"])


## TASK 3. Merge
Input: trained `LoRALinear`. Output: ordinary `nn.Linear` with
$$W_{merged}=W_0+(\alpha/r)BA.$$
**Hints:** use `no_grad`; add the update once; return the wrapped base layer (its bias is already correct).
Merging changes base weights permanently. Reload the pretrained model for another rank experiment.


In [ ]:
@torch.no_grad()
def merge_lora(model):
    return model


## Optional: rank experiment
For $r\in\{1,2,4,8,16\}$, reload the **same pretrained checkpoint** and repeat with the same
split, shuffle seed, number of steps, and $\alpha/r=2$. Record trainable count, paraphrase
loss/EM, unseen EM, and runtime. Never reuse merged weights as a baseline.
